In [160]:
import pandas as pd

df=pd.read_json("D:/FM/5_EMA/artifacts/hist_data_1m.json")

df

,date,time,open,high,low,close,volume,symbol
0,2026-01-12,09:15:00,619.85,624.40,619.05,623.15,318432,VEDL
1,2026-01-12,09:16:00,623.15,623.35,620.30,621.75,119177,VEDL
2,2026-01-12,09:17:00,621.75,621.75,619.20,619.75,104391,VEDL
3,2026-01-12,09:18:00,619.75,620.95,619.60,620.20,126006,VEDL
4,2026-01-12,09:19:00,620.05,620.80,618.00,618.05,109443,VEDL
...,...,...,...,...,...,...,...,...
24356,2026-04-20,15:11:00,771.90,771.95,771.40,771.80,32293,VEDL
24357,2026-04-20,15:12:00,771.65,771.80,771.10,771.35,38122,VEDL
24358,2026-04-20,15:13:00,771.35,771.40,771.15,771.30,29212,VEDL
24359,2026-04-20,15:14:00,771.30,771.35,770.70,771.05,81627,VEDL


In [161]:
df['sma_5'] = df['close'].rolling(window=5).mean()

In [162]:
df['prev_sma_5']=df['sma_5'].shift(1)

In [163]:
df.head()

,date,time,open,high,low,close,volume,symbol,sma_5,prev_sma_5
0,2026-01-12,09:15:00,619.85,624.40,619.05,623.15,318432,VEDL,NaN,NaN
1,2026-01-12,09:16:00,623.15,623.35,620.30,621.75,119177,VEDL,NaN,NaN
2,2026-01-12,09:17:00,621.75,621.75,619.20,619.75,104391,VEDL,NaN,NaN
3,2026-01-12,09:18:00,619.75,620.95,619.60,620.20,126006,VEDL,NaN,NaN
4,2026-01-12,09:19:00,620.05,620.80,618.00,618.05,109443,VEDL,620.58,NaN


In [164]:
#df = df[['date','time','close','prev_sma_5']]
df = df[(df['time'] >= '09:20:00') & (df['time'] <= '15:00:00')]

In [165]:
df.head()

,date,time,open,high,low,close,volume,symbol,sma_5,prev_sma_5
5,2026-01-12,09:20:00,618.00,618.55,617.50,617.90,65762,VEDL,619.53,620.58
6,2026-01-12,09:21:00,617.95,618.40,617.25,617.65,39445,VEDL,618.71,619.53
7,2026-01-12,09:22:00,617.65,617.95,617.00,617.85,46390,VEDL,618.33,618.71
8,2026-01-12,09:23:00,617.50,617.75,615.95,616.40,84327,VEDL,617.57,618.33
9,2026-01-12,09:24:00,616.25,616.70,615.60,615.70,80804,VEDL,617.10,617.57


In [166]:
import pandas as pd
import numpy as np

# =========================
# 🧹 PREP
# =========================
df = df.copy()
df = df.sort_values(by=['date', 'time']).reset_index(drop=True)

df['date'] = pd.to_datetime(df['date'])
df['datetime'] = pd.to_datetime(df['date'].dt.strftime('%Y-%m-%d') + ' ' + df['time'])

# =========================
# 🎯 ENTRY SIGNAL
# =========================
df['entry_signal'] = (
    (df['close'] > df['prev_sma_5']) &
    (df['close'].shift(1) <= df['prev_sma_5'].shift(1))
)

# =========================
# 💸 CHARGES FUNCTION (FYERS)
# =========================
def calculate_charges(entry_price, exit_price, quantity):
    
    buy_turnover = entry_price * quantity
    sell_turnover = exit_price * quantity
    turnover = buy_turnover + sell_turnover

    # Brokerage
    brokerage_buy = min(20, buy_turnover * 0.0003)
    brokerage_sell = min(20, sell_turnover * 0.0003)
    brokerage = brokerage_buy + brokerage_sell

    # STT
    stt = sell_turnover * 0.00025

    # Exchange
    txn = turnover * 0.0000345

    # SEBI
    sebi = turnover * 0.000001

    # Stamp duty
    stamp = buy_turnover * 0.00003

    # GST
    gst = 0.18 * (brokerage + txn)

    total_charges = brokerage + stt + txn + sebi + stamp + gst

    return total_charges

# =========================
# 🧠 TRADE ENGINE
# =========================
position = 0
entry_price = 0
stop_loss = 0
quantity = 0
cost = 0

initial_capital = 100000
capital = initial_capital

positions = []
trade_returns = []
pnl_list = []

for i in range(len(df)):
    
    row = df.loc[i]
    pnl = 0

    # ========= ENTRY =========
    if position == 0 and row['entry_signal'] and i > 0:
        entry_price = row['close']
        stop_loss = df.loc[i-1, 'low']
        
        quantity = int(capital // entry_price)
        
        if quantity > 0:
            cost = quantity * entry_price
            position = 1

    # ========= EXIT =========
    elif position == 1:
        
        exit_price = None

        # SL hit
        if row['low'] <= stop_loss:
            exit_price = stop_loss
        
        # SMA exit
        elif row['close'] < row['prev_sma_5']:
            exit_price = row['close']

        if exit_price is not None:
            exit_value = quantity * exit_price
            gross_pnl = exit_value - cost
            
            charges = calculate_charges(entry_price, exit_price, quantity)
            
            net_pnl = gross_pnl - charges
            
            capital += net_pnl
            pnl = net_pnl

            position = 0
            quantity = 0

    positions.append(position)
    trade_returns.append(pnl)
    pnl_list.append(pnl)

# =========================
# 💰 STORE RESULTS
# =========================
df['position'] = positions
df['pnl'] = pnl_list

# Equity curve
df['equity'] = initial_capital + df['pnl'].cumsum()

# =========================
# 📊 DAILY PnL
# =========================
daily_pnl = df.groupby(df['date'].dt.date)['pnl'].sum()

# =========================
# 📉 MAX DRAWDOWN
# =========================
rolling_max = df['equity'].cummax()
drawdown = df['equity'] / rolling_max - 1
max_drawdown = drawdown.min()

# =========================
# 📊 TRADE METRICS
# =========================
trades = df[df['pnl'] != 0].copy()
trades['trade_date'] = trades['date'].dt.date

total_trades = len(trades)
trades_per_day = trades.groupby('trade_date').size()

win_trades = trades[trades['pnl'] > 0]
loss_trades = trades[trades['pnl'] < 0]

win_rate = (len(win_trades) / total_trades * 100) if total_trades > 0 else 0

avg_win = win_trades['pnl'].mean() if len(win_trades) > 0 else 0
avg_loss = loss_trades['pnl'].mean() if len(loss_trades) > 0 else 0

gross_profit = win_trades['pnl'].sum()
gross_loss = abs(loss_trades['pnl'].sum())

profit_factor = gross_profit / gross_loss if gross_loss != 0 else np.nan

loss_rate = 1 - (win_rate / 100)
expectancy = (win_rate/100 * avg_win) - (loss_rate * abs(avg_loss))

trade_efficiency = trades['pnl'].mean() if total_trades > 0 else 0

# =========================
# 📦 FINAL REPORT
# =========================
final_capital = df['equity'].iloc[-1]
total_profit = final_capital - initial_capital
return_pct = (total_profit / initial_capital) * 100

print("\n===== FINAL REPORT =====")
print(f"Initial Capital: ₹{initial_capital}")
print(f"Final Capital: ₹{round(final_capital,2)}")
print(f"Total Profit: ₹{round(total_profit,2)}")
print(f"Return: {round(return_pct,2)}%")

print("\n===== RISK =====")
print("Max Drawdown:", round(max_drawdown * 100, 2), "%")

print("\n===== TRADE METRICS =====")
print("Total Trades:", total_trades)
print("Win Rate:", round(win_rate, 2), "%")
print("Avg Win (₹):", round(avg_win, 2))
print("Avg Loss (₹):", round(avg_loss, 2))
print("Profit Factor:", round(profit_factor, 2))
print("Expectancy (₹):", round(expectancy, 2))
print("Trade Efficiency (₹):", round(trade_efficiency, 2))

print("\n===== TRADES PER DAY =====")
print(trades_per_day)

print("\n===== DAILY PnL =====")
print(daily_pnl)


===== FINAL REPORT =====
Initial Capital: ₹100000
Final Capital: ₹8632.1
Total Profit: ₹-91367.9
Return: -91.37%

===== RISK =====
Max Drawdown: -91.38 %

===== TRADE METRICS =====
Total Trades: 2642
Win Rate: 16.35 %
Avg Win (₹): 129.87
Avg Loss (₹): -66.73
Profit Factor: 0.38
Expectancy (₹): -34.58
Trade Efficiency (₹): -34.58

===== TRADES PER DAY =====
trade_date
2026-01-12    37
2026-01-13    38
2026-01-14    32
2026-01-16    44
2026-01-19    44
              ..
2026-04-13    48
2026-04-15    40
2026-04-16    52
2026-04-17    40
2026-04-20    40
Length: 65, dtype: int64

===== DAILY PnL =====
date
2026-01-12   -2447.379738
2026-01-13   -2838.618134
2026-01-14    2931.621850
2026-01-16   -2657.840700
2026-01-19   -3458.279919
                 ...     
2026-04-13    -473.157636
2026-04-15    -503.666022
2026-04-16    -495.187243
2026-04-17    -287.409879
2026-04-20    -333.663107
Name: pnl, Length: 65, dtype: float64


After SL

In [167]:
import pandas as pd
import numpy as np

# =========================
# 🧹 PREP
# =========================
df = df.copy()
df = df.sort_values(by=['date', 'time']).reset_index(drop=True)

df['date'] = pd.to_datetime(df['date'])
df['datetime'] = pd.to_datetime(df['date'].dt.strftime('%Y-%m-%d') + ' ' + df['time'])

# =========================
# 🎯 SIGNALS
# =========================
df['long_entry'] = (
    (df['close'] > df['prev_sma_5']) &
    (df['close'].shift(1) <= df['prev_sma_5'].shift(1))
)

df['short_entry'] = (
    (df['close'] < df['prev_sma_5']) &
    (df['close'].shift(1) >= df['prev_sma_5'].shift(1))
)

# =========================
# 💸 CHARGES (FYERS)
# =========================
def calculate_charges(entry_price, exit_price, quantity):
    buy_turnover = entry_price * quantity
    sell_turnover = exit_price * quantity
    turnover = buy_turnover + sell_turnover

    brokerage = min(20, buy_turnover * 0.0003) + min(20, sell_turnover * 0.0003)
    stt = sell_turnover * 0.00025
    txn = turnover * 0.0000345
    sebi = turnover * 0.000001
    stamp = buy_turnover * 0.00003
    gst = 0.18 * (brokerage + txn)

    return brokerage + stt + txn + sebi + stamp + gst

# =========================
# 🧠 TRADE ENGINE
# =========================
position = 0  # 1 = long, -1 = short
entry_price = 0
quantity = 0
cost = 0

initial_capital = 100000
capital = initial_capital

positions = []
pnl_list = []

for i in range(len(df)):
    
    row = df.loc[i]
    pnl = 0

    # ========= ENTRY =========
    if position == 0 and i > 0:
        
        if row['long_entry']:
            entry_price = row['close']
            quantity = int(capital // entry_price)

            if quantity > 0:
                cost = quantity * entry_price
                position = 1

        elif row['short_entry']:
            entry_price = row['close']
            quantity = int(capital // entry_price)

            if quantity > 0:
                cost = quantity * entry_price
                position = -1

    # ========= EXIT =========
    elif position != 0:
        
        prev_low = df.loc[i-1, 'low']
        prev_high = df.loc[i-1, 'high']
        exit_price = None

        # ===== LONG EXIT =====
        if position == 1:
            if row['close'] < prev_low or row['close'] < row['prev_sma_5']:
                exit_price = row['close']
                exit_value = quantity * exit_price
                gross_pnl = exit_value - cost

        # ===== SHORT EXIT =====
        elif position == -1:
            if row['close'] > prev_high or row['close'] > row['prev_sma_5']:
                exit_price = row['close']
                exit_value = quantity * exit_price
                gross_pnl = cost - exit_value  # reverse logic

        if exit_price is not None:
            charges = calculate_charges(entry_price, exit_price, quantity)
            net_pnl = gross_pnl - charges

            capital += net_pnl
            pnl = net_pnl

            position = 0
            quantity = 0

    positions.append(position)
    pnl_list.append(pnl)

# =========================
# 💰 RESULTS
# =========================
df['position'] = positions
df['pnl'] = pnl_list
df['equity'] = initial_capital + df['pnl'].cumsum()

# =========================
# 📊 METRICS
# =========================
daily_pnl = df.groupby(df['date'].dt.date)['pnl'].sum()

rolling_max = df['equity'].cummax()
drawdown = df['equity'] / rolling_max - 1
max_drawdown = drawdown.min()

trades = df[df['pnl'] != 0].copy()
trades['trade_date'] = trades['date'].dt.date

total_trades = len(trades)
trades_per_day = trades.groupby('trade_date').size()

win_trades = trades[trades['pnl'] > 0]
loss_trades = trades[trades['pnl'] < 0]

win_rate = (len(win_trades) / total_trades * 100) if total_trades > 0 else 0

avg_win = win_trades['pnl'].mean() if len(win_trades) > 0 else 0
avg_loss = loss_trades['pnl'].mean() if len(loss_trades) > 0 else 0

gross_profit = win_trades['pnl'].sum()
gross_loss = abs(loss_trades['pnl'].sum())

profit_factor = gross_profit / gross_loss if gross_loss != 0 else np.nan

loss_rate = 1 - (win_rate / 100)
expectancy = (win_rate/100 * avg_win) - (loss_rate * abs(avg_loss))

# =========================
# 📦 REPORT
# =========================
final_capital = df['equity'].iloc[-1]

print("\n===== FINAL REPORT =====")
print(f"Initial Capital: ₹{initial_capital}")
print(f"Final Capital: ₹{round(final_capital,2)}")
print(f"Return: {round((final_capital/initial_capital - 1)*100,2)}%")

print("\nMax Drawdown:", round(max_drawdown * 100, 2), "%")

print("\n===== TRADE METRICS =====")
print("Total Trades:", total_trades)
print("Win Rate:", round(win_rate, 2), "%")
print("Profit Factor:", round(profit_factor, 2))
print("Expectancy (₹):", round(expectancy, 2))

print("\nTrades Per Day:")
print(trades_per_day)

print("\nDaily PnL:")
print(daily_pnl)


===== FINAL REPORT =====
Initial Capital: ₹100000
Final Capital: ₹4622.94
Return: -95.38%

Max Drawdown: -95.4 %

===== TRADE METRICS =====
Total Trades: 2855
Win Rate: 16.88 %
Profit Factor: 0.32
Expectancy (₹): -33.41

Trades Per Day:
trade_date
2026-01-12    40
2026-01-13    39
2026-01-14    32
2026-01-16    47
2026-01-19    48
              ..
2026-04-13    52
2026-04-15    46
2026-04-16    54
2026-04-17    44
2026-04-20    43
Length: 65, dtype: int64

Daily PnL:
date
2026-01-12   -3375.258655
2026-01-13   -3324.298009
2026-01-14   -2639.719780
2026-01-16   -3053.615743
2026-01-19   -2454.167951
                 ...     
2026-04-13    -332.196930
2026-04-15    -290.945271
2026-04-16    -363.115045
2026-04-17    -251.832442
2026-04-20    -207.212233
Name: pnl, Length: 65, dtype: float64


In [168]:
import pandas as pd
import numpy as np

# =========================
# 🧹 PREP
# =========================
df = df.copy()
df = df.sort_values(by=['date', 'time']).reset_index(drop=True)

df['date'] = pd.to_datetime(df['date'])
df['datetime'] = pd.to_datetime(df['date'].dt.strftime('%Y-%m-%d') + ' ' + df['time'])

# =========================
# 🎯 SIGNALS
# =========================
df['long_entry'] = (
    (df['close'] > df['prev_sma_5']) &
    (df['close'].shift(1) <= df['prev_sma_5'].shift(1))
)

df['short_entry'] = (
    (df['close'] < df['prev_sma_5']) &
    (df['close'].shift(1) >= df['prev_sma_5'].shift(1))
)

# =========================
# 💸 CHARGES (FYERS)
# =========================
def calculate_charges(entry_price, exit_price, qty=1):
    buy_turnover = entry_price * qty
    sell_turnover = exit_price * qty
    turnover = buy_turnover + sell_turnover

    brokerage = min(20, buy_turnover * 0.0003) + min(20, sell_turnover * 0.0003)
    stt = sell_turnover * 0.00025
    txn = turnover * 0.0000345
    sebi = turnover * 0.000001
    stamp = buy_turnover * 0.00003
    gst = 0.18 * (brokerage + txn)

    return brokerage + stt + txn + sebi + stamp + gst

# =========================
# 🧠 TRADE ENGINE
# =========================
position = 0  # 1 long, -1 short
entry_price = 0

initial_capital = 100000
capital = initial_capital

positions = []
pnl_list = []

# Track trade details
entry_prices = []
exit_prices = []
trade_dirs = []  # 1 long, -1 short

for i in range(len(df)):
    
    row = df.loc[i]
    pnl = 0

    # ========= ENTRY =========
    if position == 0 and i > 0:
        
        if row['long_entry']:
            entry_price = row['close']
            position = 1

        elif row['short_entry']:
            entry_price = row['close']
            position = -1

    # ========= EXIT =========
    elif position != 0:
        
        prev_low = df.loc[i-1, 'low']
        prev_high = df.loc[i-1, 'high']
        exit_price = None

        # LONG EXIT
        if position == 1:
            if row['close'] < prev_low or row['close'] < row['prev_sma_5']:
                exit_price = row['close']
                gross_pnl = exit_price - entry_price

        # SHORT EXIT
        elif position == -1:
            if row['close'] > prev_high or row['close'] > row['prev_sma_5']:
                exit_price = row['close']
                gross_pnl = entry_price - exit_price

        if exit_price is not None:
            charges = calculate_charges(entry_price, exit_price, qty=1)
            net_pnl = gross_pnl - charges

            capital += net_pnl
            pnl = net_pnl

            # store trade
            entry_prices.append(entry_price)
            exit_prices.append(exit_price)
            trade_dirs.append(position)

            position = 0

    positions.append(position)
    pnl_list.append(pnl)

# =========================
# 💰 STORE RESULTS
# =========================
df['position'] = positions
df['pnl'] = pnl_list
df['equity'] = initial_capital + df['pnl'].cumsum()

# =========================
# 📊 TRADE DATAFRAME
# =========================
trades = pd.DataFrame({
    'entry': entry_prices,
    'exit': exit_prices,
    'dir': trade_dirs
})

# Difference (direction-aware)
trades['diff'] = np.where(
    trades['dir'] == 1,
    trades['exit'] - trades['entry'],
    trades['entry'] - trades['exit']
)

# =========================
# 📊 DIFF METRICS
# =========================
max_diff = trades['diff'].max() if len(trades) > 0 else 0
min_diff = trades['diff'].min() if len(trades) > 0 else 0
avg_diff = trades['diff'].mean() if len(trades) > 0 else 0

# =========================
# 📊 OTHER METRICS
# =========================
daily_pnl = df.groupby(df['date'].dt.date)['pnl'].sum()

rolling_max = df['equity'].cummax()
drawdown = df['equity'] / rolling_max - 1
max_drawdown = drawdown.min()

total_trades = len(trades)

# =========================
# 📦 FINAL REPORT
# =========================
final_capital = df['equity'].iloc[-1]

print("\n===== FINAL REPORT =====")
print(f"Final Capital: ₹{round(final_capital,2)}")
print(f"Return: {round((final_capital/initial_capital - 1)*100,2)}%")

print("\nMax Drawdown:", round(max_drawdown * 100, 2), "%")

print("\nTotal Trades:", total_trades)

print("\n===== PRICE DIFFERENCE METRICS =====")
print("Max Diff:", round(max_diff, 2))
print("Min Diff:", round(min_diff, 2))
print("Avg Diff:", round(avg_diff, 2))

print("\n===== DAILY PnL =====")
print(daily_pnl)


===== FINAL REPORT =====
Final Capital: ₹97780.66
Return: -2.22%

Max Drawdown: -2.22 %

Total Trades: 2855

===== PRICE DIFFERENCE METRICS =====
Max Diff: 73.95
Min Diff: -40.55
Avg Diff: -0.03

===== DAILY PnL =====
date
2026-01-12   -27.141693
2026-01-13   -27.599276
2026-01-14   -23.513311
2026-01-16   -29.115858
2026-01-19   -24.595965
                ...    
2026-04-13   -46.098161
2026-04-15   -41.563610
2026-04-16   -60.205187
2026-04-17   -41.972074
2026-04-20   -34.874703
Name: pnl, Length: 65, dtype: float64


In [169]:
import pandas as pd
import numpy as np

# =========================
# 🧹 PREP
# =========================
df = df.copy()
df = df.sort_values(by=['date', 'time']).reset_index(drop=True)

df['date'] = pd.to_datetime(df['date'])
df['datetime'] = pd.to_datetime(df['date'].dt.strftime('%Y-%m-%d') + ' ' + df['time'])

# =========================
# 🎯 SIGNALS
# =========================
df['long_entry'] = (
    (df['close'] > df['prev_sma_5']) &
    (df['close'].shift(1) <= df['prev_sma_5'].shift(1))
)

df['short_entry'] = (
    (df['close'] < df['prev_sma_5']) &
    (df['close'].shift(1) >= df['prev_sma_5'].shift(1))
)

# =========================
# 💸 CHARGES (FYERS)
# =========================
def calculate_charges(entry_price, exit_price, qty=1):
    buy_turnover = entry_price * qty
    sell_turnover = exit_price * qty
    turnover = buy_turnover + sell_turnover

    brokerage = min(20, buy_turnover * 0.0003) + min(20, sell_turnover * 0.0003)
    stt = sell_turnover * 0.00025
    txn = turnover * 0.0000345
    sebi = turnover * 0.000001
    stamp = buy_turnover * 0.00003
    gst = 0.18 * (brokerage + txn)

    return brokerage + stt + txn + sebi + stamp + gst

# =========================
# 🧠 TRADE ENGINE
# =========================
position = 0
entry_price = 0

initial_capital = 100000
capital = initial_capital

positions = []
pnl_list = []

entry_prices = []
exit_prices = []
directions = []
trade_pnls = []

for i in range(len(df)):
    
    row = df.loc[i]
    pnl = 0

    # ===== ENTRY =====
    if position == 0 and i > 0:
        if row['long_entry']:
            entry_price = row['close']
            position = 1

        elif row['short_entry']:
            entry_price = row['close']
            position = -1

    # ===== EXIT =====
    elif position != 0:
        prev_low = df.loc[i-1, 'low']
        prev_high = df.loc[i-1, 'high']
        exit_price = None

        if position == 1:
            if row['close'] < prev_low or row['close'] < row['prev_sma_5']:
                exit_price = row['close']
                gross_pnl = exit_price - entry_price

        elif position == -1:
            if row['close'] > prev_high or row['close'] > row['prev_sma_5']:
                exit_price = row['close']
                gross_pnl = entry_price - exit_price

        if exit_price is not None:
            charges = calculate_charges(entry_price, exit_price, qty=1)
            net_pnl = gross_pnl - charges

            capital += net_pnl
            pnl = net_pnl

            # store trade
            entry_prices.append(entry_price)
            exit_prices.append(exit_price)
            directions.append(position)
            trade_pnls.append(net_pnl)

            position = 0

    positions.append(position)
    pnl_list.append(pnl)

# =========================
# 💰 RESULTS
# =========================
df['position'] = positions
df['pnl'] = pnl_list
df['equity'] = initial_capital + df['pnl'].cumsum()

# =========================
# 📊 TRADES DF
# =========================
trades = pd.DataFrame({
    'entry': entry_prices,
    'exit': exit_prices,
    'dir': directions,
    'pnl': trade_pnls
})

# Diff (pure price move)
trades['diff'] = np.where(
    trades['dir'] == 1,
    trades['exit'] - trades['entry'],
    trades['entry'] - trades['exit']
)

# =========================
# 📊 METRICS
# =========================
total_trades = len(trades)

win_trades = trades[trades['pnl'] > 0]
loss_trades = trades[trades['pnl'] < 0]

win_rate = (len(win_trades) / total_trades * 100) if total_trades > 0 else 0

avg_win = win_trades['pnl'].mean() if len(win_trades) > 0 else 0
avg_loss = loss_trades['pnl'].mean() if len(loss_trades) > 0 else 0

gross_profit = win_trades['pnl'].sum()
gross_loss = abs(loss_trades['pnl'].sum())

profit_factor = gross_profit / gross_loss if gross_loss != 0 else np.nan

loss_rate = 1 - (win_rate / 100)
expectancy = (win_rate/100 * avg_win) - (loss_rate * abs(avg_loss))

trade_efficiency = trades['pnl'].mean() if total_trades > 0 else 0

# =========================
# 📉 DRAWDOWN
# =========================
rolling_max = df['equity'].cummax()
drawdown = df['equity'] / rolling_max - 1
max_drawdown = drawdown.min()

# =========================
# 📊 SHARPE
# =========================
returns = df['pnl']

if returns.std() != 0:
    sharpe = (returns.mean() / returns.std()) * np.sqrt(252 * 375)
else:
    sharpe = 0

# =========================
# 📊 DAILY METRICS
# =========================
daily_pnl = df.groupby(df['date'].dt.date)['pnl'].sum()
trades_per_day = trades.groupby(df.loc[df['pnl'] != 0, 'date'].dt.date).size()

# =========================
# 📊 DIFF METRICS
# =========================
max_diff = trades['diff'].max() if total_trades > 0 else 0
min_diff = trades['diff'].min() if total_trades > 0 else 0
avg_diff = trades['diff'].mean() if total_trades > 0 else 0

# =========================
# 📦 FINAL REPORT
# =========================
final_capital = df['equity'].iloc[-1]
total_profit = final_capital - initial_capital
return_pct = (total_profit / initial_capital) * 100

print("\n===== FINAL REPORT =====")
print(f"Initial Capital: ₹{initial_capital}")
print(f"Final Capital: ₹{round(final_capital,2)}")
print(f"Total Profit: ₹{round(total_profit,2)}")
print(f"Return: {round(return_pct,2)}%")

print("\n===== RISK =====")
print("Max Drawdown:", round(max_drawdown * 100, 2), "%")
print("Sharpe Ratio:", round(sharpe, 2))

print("\n===== TRADE METRICS =====")
print("Total Trades:", total_trades)
print("Win Rate:", round(win_rate, 2), "%")
print("Avg Win (₹):", round(avg_win, 2))
print("Avg Loss (₹):", round(avg_loss, 2))
print("Profit Factor:", round(profit_factor, 2))
print("Expectancy (₹):", round(expectancy, 2))
print("Trade Efficiency (₹):", round(trade_efficiency, 2))

print("\n===== PRICE DIFFERENCE METRICS =====")
print("Max Diff:", round(max_diff, 2))
print("Min Diff:", round(min_diff, 2))
print("Avg Diff:", round(avg_diff, 2))

print("\n===== TRADES PER DAY =====")
print(trades_per_day)

print("\n===== DAILY PnL =====")
print(daily_pnl)


===== FINAL REPORT =====
Initial Capital: ₹100000
Final Capital: ₹97780.66
Total Profit: ₹-2219.34
Return: -2.22%

===== RISK =====
Max Drawdown: -2.22 %
Sharpe Ratio: -34.59

===== TRADE METRICS =====
Total Trades: 2855
Win Rate: 16.71 %
Avg Win (₹): 1.67
Avg Loss (₹): -1.27
Profit Factor: 0.26
Expectancy (₹): -0.78
Trade Efficiency (₹): -0.78

===== PRICE DIFFERENCE METRICS =====
Max Diff: 73.95
Min Diff: -40.55
Avg Diff: -0.03

===== TRADES PER DAY =====
date
2026-01-12    40
2026-01-13    39
2026-01-14    32
2026-01-16    47
2026-01-19    48
2026-01-20    46
2026-01-21    47
2026-01-22    48
2026-01-23    16
dtype: int64

===== DAILY PnL =====
date
2026-01-12   -27.141693
2026-01-13   -27.599276
2026-01-14   -23.513311
2026-01-16   -29.115858
2026-01-19   -24.595965
                ...    
2026-04-13   -46.098161
2026-04-15   -41.563610
2026-04-16   -60.205187
2026-04-17   -41.972074
2026-04-20   -34.874703
Name: pnl, Length: 65, dtype: float64
